In [0]:
import json, sys, joblib, os, time
from databricks.sdk import WorkspaceClient

In [0]:
# Databricks widgets
dbutils.widgets.text("JOB_ID_S", "", "Job ID (Small cluster)")
dbutils.widgets.text("JOB_ID_M", "", "Job ID (Medium cluster)")
dbutils.widgets.text("JOB_ID_L", "", "Job ID (Large cluster)")
dbutils.widgets.dropdown("DEFAULT_CLUSTER", "M", ["S", "M", "L"], "Default cluster")

In [0]:
# Global switches passed to child
dbutils.widgets.dropdown("MCH_DISABLE_DM", "1", ["0", "1"], "Disable DM (1=yes)")
dbutils.widgets.text("MCH_PREFILTER_TOPK", "200", "Prefilter TOP-K")
dbutils.widgets.text("MCH_RF_N_JOBS", "1", "RF n_jobs")
dbutils.widgets.text("MCH_CV_N_JOBS", "1", "CV n_jobs")
dbutils.widgets.dropdown("MCH_SAVE_MODE", "mlflow-only", ["nosave", "mlflow-only", "joblib"], "Save mode")

In [0]:
JOBLIB_PATH = "/Workspace/9900-f18a-cake/working_branch/data/freeze0525/diseaseTree_mapped.joblib"

In [0]:
sys.path.append("/Workspace/9900-f18a-cake/working_branch//src")
print("Loading DiseaseTree to get the list of nodes...")

try:
    tree_object = joblib.load(JOBLIB_PATH)
    print("DiseaseTree loaded successfully.")
except Exception as e:
    print(f"Failed to load DiseaseTree: {e}")
    tree_object = None

def get_nodes_to_train(tree):
    """
    Traverses the tree object to extract a list of all node names.
    """
    all_nodes = []
    if tree is None: return []
    
    def traverse(node):
        # Only include non-root nodes that have samples
        if node.name != 'ZERO2' and hasattr(node, 'samples') and len(node.samples) > 0:
             all_nodes.append(node.name)
        # Recurse into children
        if hasattr(node, 'children'):
            for child in node.children:
                traverse(child)
    
    traverse(tree)
    return list(set(all_nodes))

# Get the full list of nodes from the file
nodes_to_train = get_nodes_to_train(tree_object)
print(nodes_to_train)

Loading DiseaseTree to get the list of nodes...
DiseaseTree loaded successfully.
['Colorectal carcinoma', 'Medullary thyroid carcinoma', 'Olfactory neuroblastoma', 'CIC-rearranged sarcoma, non-CNS', 'Malignant peripheral nerve sheath tumour', 'Adrenocortical carcinoma', 'Malignant rhabdoid tumour, liver', 'T cell lymphoproliferative disorder', 'Astrocytoma, IDH-mutant WHO G3', 'Medulloblastoma, SHH-activated and TP53-wildtype', 'Pineal parenchymal tumour of intermediate differentiation, WHO G3', 'Embryonal rhabdomyosarcoma', 'Hepatosplenic T-cell lymphoma', 'Renal cell carcinoma', 'Posterior fossa A ependymoma, group PFA, WHO G3', 'PIK3CA-related overgrowth tumour', 'Clear cell sarcoma of kidney', 'Gastrointestinal stromal tumour', 'CNS embryonal tumour with PLAG-family alteration', 'EBV-associated smooth muscle tumour', 'Acute myeloid leukaemia', 'DICER1-associated sarcoma', 'Neuro-epithelial tumour with PATZ1 fusion', 'Malignant rhabdoid tumour, non-renal/hepatic', 'BCOR-rearranged s

In [0]:
#Manual Node Selection
default_nodes = [
    {"node_id": "Haematological malignancy", "cluster": "M"},
    # {"node_id": "Sarcoma", "cluster": "L"},
    # {"node_id": "Melanoma", "cluster": "S"},
]

In [0]:
#Auto Node Selection
#nodes_to_run = nodes_to_train

In [0]:
dbutils.widgets.text("NODE_LIST_JSON", json.dumps(default_nodes), "Nodes (JSON list)")

In [0]:
JOB_ID_S = dbutils.widgets.get("JOB_ID_S").strip()
JOB_ID_M = dbutils.widgets.get("JOB_ID_M").strip()
JOB_ID_L = dbutils.widgets.get("JOB_ID_L").strip()
DEFAULT_CLUSTER = dbutils.widgets.get("DEFAULT_CLUSTER").strip().upper()

MCH_DISABLE_DM = dbutils.widgets.get("MCH_DISABLE_DM").strip()
MCH_PREFILTER_TOPK = dbutils.widgets.get("MCH_PREFILTER_TOPK").strip()
MCH_RF_N_JOBS = dbutils.widgets.get("MCH_RF_N_JOBS").strip()
MCH_CV_N_JOBS = dbutils.widgets.get("MCH_CV_N_JOBS").strip()
MCH_SAVE_MODE = dbutils.widgets.get("MCH_SAVE_MODE").strip()

In [0]:
try:
    NODES = json.loads(dbutils.widgets.get("NODE_LIST_JSON"))
    assert isinstance(NODES, list)
except Exception as e:
    raise ValueError("NODE_LIST_JSON must be a JSON list of objects like "
                     '[{"node_id":"X","cluster":"M"}]') from e

In [0]:
# cluster to job_id map 
def _as_int_or_zero(s): 
    try: return int(s)
    except: return 0

In [0]:
CLUSTER_TO_JOB = {
    "S": _as_int_or_zero(JOB_ID_S),
    "M": _as_int_or_zero(JOB_ID_M),
    "L": _as_int_or_zero(JOB_ID_L),
}

In [0]:
missing = [k for k,v in CLUSTER_TO_JOB.items() if v == 0]
if missing:
    print("Missing Job IDs for:", missing)
    print("Fill JOB_ID_S / JOB_ID_M / JOB_ID_L widgets before launching.")

In [0]:
def build_child_params(node_id: str) -> dict:
    # Child notebook reads these as notebook params; it can export them to env if needed
    return {
        "NODE_ID": node_id,
        "MCH_ONLY_NODE": node_id,
        "MCH_DISABLE_DM": MCH_DISABLE_DM,          # "1" or "0"
        "MCH_PREFILTER_TOPK": MCH_PREFILTER_TOPK,  # "200"
        "MCH_RF_N_JOBS": MCH_RF_N_JOBS,            # "1"
        "MCH_CV_N_JOBS": MCH_CV_N_JOBS,            # "1"
        "MCH_SAVE_MODE": MCH_SAVE_MODE,            # "mlflow-only" | "joblib" | "nosave"
    }

In [0]:
w = WorkspaceClient()

In [0]:
submitted = []

for item in NODES:
    node_id = item.get("node_id")
    if not node_id:
        print("Skipping an item without 'node_id':", item)
        continue

    cluster_key = (item.get("cluster") or DEFAULT_CLUSTER).upper()
    if cluster_key not in CLUSTER_TO_JOB:
        raise ValueError(f"Unknown cluster key '{cluster_key}'. Use one of {list(CLUSTER_TO_JOB)}")

    job_id = CLUSTER_TO_JOB[cluster_key]
    if not job_id:
        raise ValueError(f"No Job ID configured for cluster '{cluster_key}'. "
                         f"Set the widget JOB_ID_{cluster_key}.")

    params = build_child_params(node_id)
    print(f"\nSubmitting: node='{node_id}' on cluster='{cluster_key}' via job_id={job_id}")
    print(f"Params: {params}")

    rn = w.jobs.run_now(job_id=job_id, notebook_params=params)
    
    submitted.append((node_id, cluster_key, rn.run_id))
    print(f"→ run_id={rn.run_id}")


Submitting: node='Haematological malignancy' on cluster='M' via job_id=1114821904435730
Params: {'NODE_ID': 'Haematological malignancy', 'MCH_ONLY_NODE': 'Haematological malignancy', 'MCH_DISABLE_DM': '1', 'MCH_PREFILTER_TOPK': '200', 'MCH_RF_N_JOBS': '1', 'MCH_CV_N_JOBS': '1', 'MCH_SAVE_MODE': 'mlflow-only'}
→ run_id=804842230688758


In [0]:
print("\n=== Summary ===")
for node_id, ck, run_id, url in submitted:
    print(f"{node_id:35s} | {ck} | {run_id})

  File <command-6371680499939704>, line 3
    print(f"{node_id:35s} | {ck} | {run_id})
          ^
SyntaxError: unterminated f-string literal (detected at line 3)
